<img src="https://drive.google.com/uc?export=view&id=1AQr9H9bXDeNPchTRufU78g8z0yxHvrmC" width="100%">


**Grupo 12**

David Martínez Triana · Andrés Ramírez Romero · David Delgado Ortiz

# Entendimiento del Negocio y Carga de Textos
---

**Proyecto aplicado — Fase 1** (10% del proyecto)

**Proyecto:** recuperación de la respuesta oficial para un asistente de preguntas frecuentes de la Universidad Nacional de Colombia.

Esta entrega deja el marco del proyecto y el corpus que usa la fase 2:

- quién consulta y cuál es el problema;
- objetivo del negocio y objetivo del sistema;
- planeación de las cuatro entregas;
- las páginas de donde salen los textos y el código que las lee.


### **Dónde queda esta fase**

```mermaid
flowchart LR
  F1["Fase 1<br/>Marco y corpus"] --> F2["Fase 2<br/>Exploración y limpieza"]
  F2 --> F3["Fase 3<br/>Representación"]
  F3 --> F4["Fase 4<br/>Recuperación"]
  F1 -.-> CSV["corpus_faq_unal.csv"]
  CSV -.-> F2
```

Esta entrega cierra el marco y deja el CSV. Las fases siguientes no vuelven a scrapear: leen el archivo.


## **1. Marco de Proyecto**
---


### **1.1. Trasfondo del negocio**
---

- ¿Quién es el cliente o los beneficiarios del proyecto? ¿En qué dominio se encuentran?
- ¿Qué problemas del negocio o del dominio estamos tratando de solucionar?

El beneficiario es la persona de la comunidad universitaria —aspirante, estudiante, egresado o funcionario— que busca, en español, una respuesta que la Universidad Nacional de Colombia ya publicó. El dominio es la atención institucional.

La Universidad no tiene una sola lista de preguntas frecuentes. Cada facultad, biblioteca, oficina de personal, sistema de investigación y sede publica la suya. Quien pregunta tiene que saber cuál sitio abrir. Esta entrega reúne las páginas que el equipo seleccionó: Ingeniería (la FAQ de la facultad y la de la UECP), Estadística, Medicina, Humanas, bibliotecas, Hermes, SIAPAL, admisiones y registro donde la página sigue en línea, secretaría, veeduría, DAMA Manizales, movilidad vehicular y el glosario de la UECP.

El problema no es inventar una respuesta. Es encontrar el texto oficial que corresponde a la pregunta y mostrar de qué página salió. Varias URL de la lista ya no responden (error 404 o 503) o no traen el listado dentro del HTML. Esas páginas se informan y no se reemplazan con texto escrito a mano.


### **1.2. Alcance**
---

- ¿Qué solución de procesamiento de lenguaje natural tratamos de implementar?
- ¿Qué se hará?
- ¿De qué forma el cliente o beneficiario utilizará el producto del proyecto?

**Objetivo del negocio.** Devolver la respuesta publicada por la Universidad y dejar visible la página de origen.

**Objetivo del proyecto.** Construir un sistema de recuperación de preguntas frecuentes en español. Ante una pregunta, busca en el corpus el par más pertinente y devuelve el texto publicado, con la fuente, la URL y, cuando la página lo trae, la sección.

El trabajo sigue las cuatro entregas:

1. **Esta fase** arma el corpus desde las páginas oficiales que sí se pueden leer.
2. **Fase 2** caracteriza ese corpus —está mezclado y no todas las fuentes pesan igual— y prepara el texto.
3. **Fase 3** representa preguntas y respuestas para poder compararlas.
4. **Fase 4** recupera la respuesta y compara configuraciones con métricas de ranking.

**Uso.** La persona escribe su pregunta. El sistema muestra la respuesta oficial más cercana y la referencia de la página. No redacta una respuesta nueva.

El corpus mezcla temas distintos: trámites de facultad, bibliotecas, proyectos de Hermes, nómina y un glosario. Por eso cada fila guarda `fuente` y `url`. En las fases siguientes se puede filtrar o ponderar, en lugar de tratar todas las páginas como si fueran la misma.


### **1.3. Plan**
---

| Entrega | Actividad | Duración | Entregable | Criterio de cierre |
|---|---|---|---|---|
| 1. Entendimiento del negocio y carga | Definir el problema y leer las páginas de FAQ seleccionadas | 1 semana | Este notebook y `corpus_faq_unal.csv` | Cada fila tiene pregunta, respuesta, fuente y URL; las páginas caídas quedan reportadas |
| 2. Entendimiento y preparación del corpus | Explorar fuentes, longitudes y secciones; normalizar el texto | 1 semana | Notebook de exploración y corpus preparado | Los hallazgos describen el desbalance y el texto queda listo para representarlo |
| 3. Extracción de características | Elegir y construir la representación de preguntas y respuestas | 1 semana | Notebook de características | La representación sirve para comparar una pregunta nueva con el corpus |
| 4. Modelamiento y evaluación | Recuperar la respuesta, comparar configuraciones y medir el ranking | 1 semana | Notebook de modelo, comparación y métricas | La comparación muestra qué configuración recupera mejor la respuesta oficial |

Dos riesgos ya vistos en esta carga: varias URL oficiales ya no existen, y el HTML no tiene la misma estructura en todas las sedes. El archivo `corpus_faq_unal.csv` deja el resultado de esta lectura para que la fase 2 no dependa de que las 34 páginas respondan otra vez.


## **2. Definición del Corpus**
---


### **2.1. Origen**
---

- ¿De dónde vienen los textos?
- ¿Se usa alguna herramienta o proceso para la descarga de la información textual?
- ¿Qué tipo de textos estamos manejando?

Los textos son pares en español publicados por la Universidad: preguntas frecuentes y, en una página, definiciones de un glosario. Salen de la lista de URL del equipo. El código abre cada página con `requests`, lee el HTML con `BeautifulSoup` y se queda con la estrategia que más pares válidos devuelve:

- el marcado `FAQPage` de schema.org, cuando la página lo trae (Ingeniería y el glosario de la UECP);
- el acordeón de Estadística y el de Hermes;
- las parejas `span.pregunta` / `span.respuesta` de SIAPAL;
- los bloques `<details>` y, si no, los encabezados o párrafos que son una pregunta seguidos de su respuesta;
- en el índice de Bibliotecas, un paso adicional: entra a cada sección enlazada desde esa página.

No hace falta un navegador. La convocatoria abierta de 2018 arma la FAQ con JavaScript y el HTML estático llega vacío; esa página queda reportada, no inventada.

Cada fila tiene cinco columnas: `pregunta`, `respuesta`, `fuente`, `url` y `titulo_seccion`. Si la página no agrupa por sección, esa columna va vacía. El glosario usa el término como `pregunta` y la definición como `respuesta`, con sección `Glosario`.

`movilidad.unal.edu.co` no es la movilidad académica: es el registro de vehículos. La movilidad de estudiantes está en ORI Medellín y en Humanas.


### **2.2. Carga o adquisición de textos**
---

El código lee las 34 páginas, separa las secciones que venían pegadas en un solo bloque y guarda solo las cinco columnas en `corpus_faq_unal.csv`. La revisión de la celda siguiente no se escribe en el archivo.

Requisito: `pandas`, `requests` y `beautifulsoup4`. DAMA Manizales presenta un certificado que esta red no valida; esa única sede se lee sin verificar el certificado, y el resto sí se verifica.


In [1]:
import importlib
import subprocess
import sys

_faltan = []
for _modulo, _paquete in [("pandas", "pandas"), ("requests", "requests"), ("bs4", "beautifulsoup4")]:
    try:
        importlib.import_module(_modulo)
    except ImportError:
        _faltan.append(_paquete)
if _faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_faltan])
    print("Instalados:", ", ".join(_faltan))
else:
    print("Dependencias listas: pandas, requests, beautifulsoup4.")


Dependencias listas: pandas, requests, beautifulsoup4.


### **Cómo se arma el CSV**

```mermaid
flowchart TD
  U["Lista de URLs"] --> R["recolectar"]
  R -->|OK| E["extraer_pagina"]
  R -->|falla| Z["anotar y seguir"]
  E --> S{"Estrategia"}
  S --> A["Estadística / SIAPAL / Hermes"]
  S --> B["JSON FAQPage"]
  S --> C["details o bloques"]
  A --> F["refinar"]
  B --> F
  C --> F
  F --> M["main"]
  M --> CSV["corpus_faq_unal.csv<br/>pregunta · respuesta · fuente · url · titulo_seccion"]
  M --> X["páginas en cero reportadas"]
```

Una sola estrategia por página. Bibliotecas además entra a las secciones del índice. No se inventan filas para las URL caídas.


El código de la celda siguiente hace, en este orden:

1. `recolectar` abre cada URL. Si falla, la anota y sigue con la siguiente.
2. `extraer_pagina` elige una sola estrategia por página: Estadística, SIAPAL, Hermes, el JSON `FAQPage`, los `<details>` o los bloques de pregunta y respuesta.
3. En Bibliotecas entra además a cada sección enlazada desde el índice.
4. `refinar` parte los bloques que traían varias preguntas pegadas y deja fuera los títulos que no son pregunta ni término de glosario.
5. `main` guarda `corpus_faq_unal.csv` con las cinco columnas e imprime las páginas que quedaron en cero.


In [2]:
# -*- coding: utf-8 -*-
"""Extrae pares pregunta-respuesta de las FAQ indicadas por el equipo."""
import json
import re
import warnings
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", message="Unverified HTTPS request")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
}
COLUMNAS = ["pregunta", "respuesta", "fuente", "url", "titulo_seccion"]
PREGUNTA_RE = re.compile(r"¿[^?]{8,280}\?")
SOLO_PREGUNTA_RE = re.compile(r"^(?:\d+\.\s*)?¿[^?]{8,280}\?$")

# (nombre corto, url, titulo por defecto)
FUENTES = [
    ("Hermes", "http://www.hermes.unal.edu.co/pages/Consultas/preguntas.xhtml", ""),
    ("Ingeniería UECP — FAQ", "https://ingenieria.bogota.unal.edu.co/uecp/index.php/preguntas-frecuentes/faq", ""),
    ("SIAPAL", "https://siapal.unal.edu.co/academia/libre-acceso/faq.do", ""),
    ("Ingeniería UECP — Glosario", "https://ingenieria.bogota.unal.edu.co/uecp/index.php/preguntas-frecuentes/glosario", "Glosario"),
    ("Convocatoria abierta 2018", "https://concabiertoca2018.unal.edu.co/#/faq", ""),
    ("Bibliotecas — índice", "https://bibliotecas.unal.edu.co/formacion-y-asistencia/preguntas-frecuentes", ""),
    ("Ingeniería Bogotá", "https://ingenieria.bogota.unal.edu.co/es/preguntas-frecuentes/", ""),
    ("Admisiones pregrado", "https://admisiones.unal.edu.co/pregrado/preguntas-frecuentes-pregrado/", ""),
    ("ORI Medellín", "https://ori.medellin.unal.edu.co/preguntas-frecuentes.html", ""),
    ("Minas — trámites", "https://minas.medellin.unal.edu.co/tramitesestudiantiles/preguntas-frecuentes-faq-s.html", ""),
    ("Admisiones posgrado", "https://admisiones.unal.edu.co/posgrado/preguntas-frecuentes/", ""),
    ("Minas — prácticas", "https://minas.medellin.unal.edu.co/informacion-de-practicas/preguntas-frecuentes", ""),
    ("Quejas y reclamos", "https://quejasyreclamos.unal.edu.co/preguntas_frecuentes.php", ""),
    ("Bibliotecas sede Bogotá", "https://bibliotecas.unal.edu.co/sede-bogota/servicios/servicios-en-la-sede/preguntas-frecuentes", ""),
    ("Ingeniería PRI", "https://ingenieria.bogota.unal.edu.co/pri/index.php/home/pri-preguntas-frecuentes", ""),
    ("Secretaría General", "https://secretaria.unal.edu.co/index.php?id=81", ""),
    ("Humanas — internacionalización", "https://www.humanas.unal.edu.co/2017/facultad/internacionalizacion/faq", ""),
    ("Bibliotecas — CLIC", "https://bibliotecas.unal.edu.co/sede-bogota/servicios/formacion/preguntas-frecuentes-clic", ""),
    ("Humanas — Bienestar", "https://www.humanas.unal.edu.co/2017/bienestar/preguntas-frecuentes", ""),
    ("Egresados Medellín", "https://medellin.unal.edu.co/egresados/egresados/preguntas-frecuentes.html", ""),
    ("Educación continua Medellín", "https://medellin.unal.edu.co/educacioncontinua/preguntas-frecuentes.html", ""),
    ("Personal académico", "https://direccionpersonal.bogota.unal.edu.co/preguntas-frecuentes-personal-academico/", ""),
    ("Arquitectura — documentación", "https://arquitectura.medellin.unal.edu.co/escuelas/habitat/index.php/13-preguntas-frecuentes/260-preguntas-frecuentes-de-la-unidad-de-documentacion", ""),
    ("Dirección de Personal", "https://direccionpersonal.bogota.unal.edu.co/preguntas-frecuentes/", ""),
    ("Dirección Académica Bogotá", "https://diracad.bogota.unal.edu.co/formacion-integral/informacion-general/preguntas-frecuentes", ""),
    ("División salarial", "https://direccionpersonal.bogota.unal.edu.co/preguntas-frecuentes-division-salarial/", ""),
    ("DAMA Manizales", "https://dama.manizales.unal.edu.co/index.php/preguntas-frecuentes/", ""),
    ("Gestión vehicular", "https://movilidad.unal.edu.co/preguntas-frecuentes.html", ""),
    ("Registro y matrícula Medellín", "https://registroymatricula.medellin.unal.edu.co/preguntas-frecuentes-estudiantes-y-admitidos-de-posgrado.html", ""),
    ("Medicina Bogotá", "https://medicina.bogota.unal.edu.co/facultad/preguntas-frecuentes", ""),
    ("Explora UN Mundo", "https://www.humanas.unal.edu.co/explora/explora-un-mundo", ""),
    ("Egresados UNAL", "https://egresados.unal.edu.co/index.php/preguntas-frecuentes/preguntas-egresados", ""),
    ("Estadística", "https://ciencias.bogota.unal.edu.co/areas_curriculares/estadistica/preguntas_frecuentes/", ""),
    ("Veeduría disciplinaria Medellín", "https://veeduriadisciplinaria.medellin.unal.edu.co/index.php/preguntas-frecuentes", ""),
]


def normalizar(texto):
    if texto is None or isinstance(texto, float):
        return ""
    return " ".join(str(texto).replace("\xa0", " ").split())


def limpiar_html(texto):
    if not texto:
        return ""
    if "<" in texto and ">" in texto:
        return normalizar(BeautifulSoup(texto, "html.parser").get_text(" ", strip=True))
    return normalizar(texto)


def fila(pregunta, respuesta, fuente, url, titulo=""):
    return {
        "pregunta": normalizar(pregunta),
        "respuesta": normalizar(respuesta),
        "fuente": fuente,
        "url": url,
        "titulo_seccion": normalizar(titulo),
    }


def valida(par):
    p, r = par["pregunta"], par["respuesta"]
    if r.lower().startswith(p.lower()):
        r = r[len(p):].strip(" .:-")
    r = re.sub(r"^(RTA|Respuesta)\s*[:/]\s*", "", r, flags=re.I).strip()
    par["respuesta"] = r
    minimo = 12 if par.get("titulo_seccion") == "Glosario" else 20
    if len(p) < 8 or len(r) < minimo:
        return False
    if p.lower() == r.lower():
        return False
    # Un menú de navegación no trae puntuación; una respuesta sí.
    if par.get("titulo_seccion") != "Glosario" and not re.search(r"[.;:]", r) and len(r) < 180:
        return False
    if len(r) > 6000:
        par["respuesta"] = r[:6000].rsplit(" ", 1)[0]
    return True


def refinar(filas):
    """Parte las secciones que venían pegadas y descarta títulos que no son pregunta ni glosario."""
    salida = []
    for par in filas:
        respuesta = par["respuesta"]
        pregunta = par["pregunta"]
        es_pregunta = ("¿" in pregunta) or pregunta.rstrip().endswith("?")
        if not es_pregunta and par.get("titulo_seccion") != "Glosario":
            coincidencias = list(PREGUNTA_RE.finditer(respuesta))
            if coincidencias:
                titulo_previo = normalizar(par.get("titulo_seccion"))
                titulo = (titulo_previo or pregunta)[:120]
                for indice, coincidencia in enumerate(coincidencias):
                    fin = coincidencias[indice + 1].start() if indice + 1 < len(coincidencias) else len(respuesta)
                    texto = respuesta[coincidencia.end():fin].strip(" .-")
                    salida.append(fila(coincidencia.group(0), texto, par["fuente"], par["url"], titulo))
                continue
            if respuesta.strip().lower().startswith("http") and len(respuesta) < 220:
                continue
            if not re.search(r"[.;]", respuesta):
                continue
        else:
            repetida = PREGUNTA_RE.match(respuesta)
            if repetida and repetida.group(0).casefold()[:50] == pregunta.casefold()[:50]:
                par = dict(par)
                par["respuesta"] = respuesta[repetida.end():].strip(" .-")
        salida.append(par)
    return dedup(salida)


def dedup(filas):
    vistos = set()
    salida = []
    for par in filas:
        if not valida(par):
            continue
        par["titulo_seccion"] = normalizar(par.get("titulo_seccion"))
        par["pregunta"] = normalizar(par.get("pregunta"))
        par["respuesta"] = normalizar(par.get("respuesta"))
        clave = (par["pregunta"].casefold(), par["respuesta"][:180].casefold(), par["url"], par["titulo_seccion"].casefold())
        if clave in vistos:
            continue
        vistos.add(clave)
        salida.append(par)
    return salida


def raiz_contenido(soup):
    for etiqueta in soup(["script", "style", "noscript"]):
        etiqueta.decompose()
    for selector in (
        "main",
        "article",
        "[itemprop=articleBody]",
        ".item-page",
        ".com-content-article__body",
        ".entry-content",
        "#content",
    ):
        nodo = soup.select_one(selector)
        if nodo and len(nodo.get_text(" ", strip=True)) > 400:
            return nodo
    return soup.body or soup


def desde_jsonld(soup, fuente, url, titulo):
    filas = []
    for script in soup.find_all("script", type="application/ld+json"):
        bruto = script.string or script.get_text()
        if not bruto or "FAQPage" not in bruto:
            continue
        limpio = re.sub(r"[\x00-\x1f]", " ", bruto)
        try:
            datos = json.loads(limpio)
        except json.JSONDecodeError:
            continue
        bloques = datos if isinstance(datos, list) else [datos]
        for bloque in bloques:
            if not isinstance(bloque, dict):
                continue
            if bloque.get("@type") != "FAQPage":
                grafos = bloque.get("@graph") or []
                bloques.extend(g for g in grafos if isinstance(g, dict))
                continue
            for entidad in bloque.get("mainEntity") or []:
                if not isinstance(entidad, dict):
                    continue
                pregunta = limpiar_html(entidad.get("name"))
                respuesta = limpiar_html((entidad.get("acceptedAnswer") or {}).get("text"))
                filas.append(fila(pregunta, respuesta, fuente, url, titulo))
    return dedup(filas)


def desde_details(nodo, fuente, url, titulo):
    filas = []
    for bloque in nodo.select("details"):
        resumen = bloque.find("summary")
        if resumen is None:
            continue
        pregunta = normalizar(resumen.get_text(" ", strip=True)).replace("▼", "").strip()
        respuesta = normalizar(bloque.get_text(" ", strip=True))
        if pregunta and respuesta.startswith(pregunta):
            respuesta = respuesta[len(pregunta):].strip()
        respuesta = re.sub(r"^(RTA|Respuesta)\s*[:/]\s*", "", respuesta, flags=re.I)
        filas.append(fila(pregunta, respuesta, fuente, url, titulo))
    return dedup(filas)


def desde_siapal(nodo, fuente, url, titulo):
    filas = []
    for pregunta_el in nodo.select("span.pregunta"):
        respuesta_el = pregunta_el.find_next("span", class_="respuesta")
        if respuesta_el is None:
            continue
        filas.append(fila(
            pregunta_el.get_text(" ", strip=True),
            respuesta_el.get_text(" ", strip=True),
            fuente, url, titulo,
        ))
    return dedup(filas)


def desde_ciencias(nodo, fuente, url):
    titulo = ""
    portada = {
        "Preguntas Frecuentes Estadística",
        "Preguntas frecuentes Área Curricular de Estadística",
    }
    filas = []
    for elemento in nodo.find_all(["h1", "h2", "h3", "div"]):
        if elemento.name in ("h1", "h2", "h3"):
            texto = normalizar(elemento.get_text(" ", strip=True))
            if texto.lower().startswith("preguntas"):
                titulo = texto
            continue
        if "faq-acordeon-facultad-ciencias" not in (elemento.get("class") or []):
            continue
        if not titulo or titulo in portada:
            continue
        cabeza = elemento.select_one("div.head-acordeon-facultad-ciencias")
        contenido = elemento.select_one("div.content-acordeon-facultad-ciencias")
        if cabeza is None or contenido is None:
            continue
        icono = cabeza.select_one(".toggle-icon")
        if icono is not None:
            icono.extract()
        filas.append(fila(
            cabeza.get_text(" ", strip=True),
            contenido.get_text(" ", strip=True),
            fuente, url, titulo,
        ))
    return dedup(filas)


def desde_hermes(nodo, fuente, url):
    filas = []
    for encabezado in nodo.find_all("h3"):
        pregunta = normalizar(encabezado.get_text(" ", strip=True))
        if "¿" not in pregunta and "?" not in pregunta:
            continue
        hermano = encabezado.find_next_sibling()
        if hermano is None:
            continue
        respuesta = normalizar(hermano.get_text(" ", strip=True))
        # Hermes no publica un título de sección estable: el texto anterior al acordeón es otra respuesta.
        filas.append(fila(pregunta, respuesta, fuente, url, ""))
    return dedup(filas)


BLOQUES = ("h2", "h3", "h4", "h5", "p", "li", "summary", "dt", "div")


def clasificar_bloque(texto):
    """Separa una pregunta del resto del mismo bloque. None si el bloque no es una pregunta."""
    coincidencia = PREGUNTA_RE.search(texto)
    if not coincidencia or coincidencia.start() > 60:
        return None, ""
    pregunta = coincidencia.group(0)
    resto = texto[coincidencia.end():].strip(" .-:;/ ")
    resto = re.sub(r"^(RTA|Respuesta)\s*[:/]\s*", "", resto, flags=re.I)
    if len(texto) > 500 and len(resto) < 20:
        return None, ""
    return pregunta, resto


def desde_bloques(nodo, fuente, url, titulo):
    """Lee bloques en orden: una pregunta y los bloques siguientes hasta la próxima pregunta."""
    hojas = []
    for elemento in nodo.find_all(BLOQUES):
        if elemento.name == "div" and elemento.find(["h2", "h3", "h4", "h5", "p", "li", "table"]):
            texto_div = normalizar(elemento.get_text(" ", strip=True))
            pregunta_div, resto_div = clasificar_bloque(texto_div)
            if not (pregunta_div and len(resto_div) < 40 and len(texto_div) < 360):
                continue
        texto = normalizar(elemento.get_text(" ", strip=True))
        if len(texto) < 12 or len(texto) > 2200:
            continue
        if any(
            len(normalizar(hijo.get_text(" ", strip=True))) >= max(12, int(len(texto) * 0.9))
            for hijo in elemento.find_all(BLOQUES)
        ):
            continue
        if hojas and hojas[-1] == texto:
            continue
        hojas.append(texto)

    filas = []
    indice = 0
    while indice < len(hojas):
        pregunta, resto = clasificar_bloque(hojas[indice])
        if not pregunta:
            indice += 1
            continue
        if len(resto) >= 20:
            filas.append(fila(pregunta, resto, fuente, url, titulo))
            indice += 1
            continue
        partes = []
        siguiente = indice + 1
        while siguiente < len(hojas) and len(partes) < 3:
            otra, _ = clasificar_bloque(hojas[siguiente])
            if otra:
                break
            partes.append(hojas[siguiente])
            if sum(len(parte) for parte in partes) > 1800:
                break
            siguiente += 1
        respuesta = normalizar(" ".join(partes))
        if len(respuesta) >= 20:
            filas.append(fila(pregunta, respuesta, fuente, url, titulo))
        indice = siguiente if partes else indice + 1
    return dedup(filas)


def extraer_pagina(soup, fuente, url, titulo=""):
    if "faq-acordeon-facultad-ciencias" in str(soup):
        ciencias = desde_ciencias(soup, fuente, url)
        if len(ciencias) >= 20:
            return ciencias
    if soup.select("span.pregunta") and soup.select("span.respuesta"):
        siapal = desde_siapal(soup, fuente, url, titulo)
        if siapal:
            return siapal
    if "hermes.unal.edu.co" in url:
        hermes = desde_hermes(soup, fuente, url)
        if len(hermes) >= 20:
            return hermes
    jsonld = desde_jsonld(soup, fuente, url, titulo)
    if len(jsonld) >= 8:
        return jsonld
    nodo = raiz_contenido(soup)
    details = desde_details(nodo, fuente, url, titulo)
    bloques = desde_bloques(nodo, fuente, url, titulo)
    candidatos = [grupo for grupo in (jsonld, details, bloques) if grupo]
    if not candidatos:
        return []
    return max(candidatos, key=len)


def obtener_soup(sesion, url, verificar=True):
    respuesta = sesion.get(url, headers=HEADERS, timeout=30, allow_redirects=True, verify=verificar)
    respuesta.raise_for_status()
    if respuesta.status_code >= 400:
        raise requests.HTTPError(f"HTTP {respuesta.status_code}")
    # Una página de error suave a veces responde 200 con título 404.
    respuesta.encoding = respuesta.apparent_encoding or "utf-8"
    soup = BeautifulSoup(respuesta.text, "html.parser")
    titulo_pagina = normalizar(soup.title.get_text(" ", strip=True) if soup.title else "")
    if re.search(r"\b404\b|no encontrada|not found|503", titulo_pagina, re.I):
        raise requests.HTTPError(f"Página de error: {titulo_pagina[:80]}")
    return soup, str(respuesta.url)


def enlaces_biblioteca(soup, url_base):
    encontrados = []
    vistos = set()
    for ancla in soup.select("a[href]"):
        href = ancla.get("href", "").strip()
        texto = normalizar(ancla.get_text(" ", strip=True))
        if not href or href.startswith("#") or not texto:
            continue
        absoluto = urljoin(url_base, href).split("#")[0].rstrip("/")
        if "preguntas-frecuentes/" not in absoluto:
            continue
        if absoluto.rstrip("/") == url_base.split("#")[0].rstrip("/"):
            continue
        if absoluto in vistos:
            continue
        vistos.add(absoluto)
        encontrados.append((texto, absoluto))
    return encontrados


def recolectar(sesion, nombre, url, titulo=""):
    verificar = "dama.manizales" not in url
    try:
        soup, url_final = obtener_soup(sesion, url, verificar=verificar)
    except Exception as exc:
        return [], f"{type(exc).__name__}: {exc}"
    if nombre == "Bibliotecas — índice":
        filas = []
        for seccion, enlace in enlaces_biblioteca(soup, url_final):
            try:
                sub, _ = obtener_soup(sesion, enlace)
            except Exception:
                continue
            filas.extend(extraer_pagina(sub, nombre, enlace, seccion))
        propias = extraer_pagina(soup, nombre, url_final, titulo)
        return dedup(filas + propias), ""
    return extraer_pagina(soup, nombre, url_final, titulo), ""


def main():
    sesion = requests.Session()
    todas = []
    reporte = []
    for nombre, url, titulo in FUENTES:
        filas, error = recolectar(sesion, nombre, url, titulo)
        reporte.append({
            "fuente": nombre,
            "url": url,
            "pares": len(filas),
            "estado": "ok" if filas else (error or "sin pares en el HTML"),
            "ejemplo": filas[0]["pregunta"][:90] if filas else "",
        })
        todas.extend(filas)
        print(f"{len(filas):4} al leer   {nombre}" + (f"   {error}" if error else ""))
    todas = refinar(todas)
    corpus = pd.DataFrame(todas, columns=COLUMNAS)
    if len(corpus):
        corpus = corpus.drop_duplicates(subset=["pregunta", "respuesta", "url"]).reset_index(drop=True)
        conteos = corpus["fuente"].value_counts()
        for item in reporte:
            item["pares"] = int(conteos.get(item["fuente"], 0))
            if item["pares"] and item["estado"] != "ok":
                item["estado"] = "ok"
            if item["pares"] == 0 and item["estado"] == "ok":
                item["estado"] = "sin pares en el HTML"
    destino = Path("corpus_faq_unal.csv")
    corpus.to_csv(destino, index=False, encoding="utf-8")
    print("Pares en el corpus, después de separar secciones pegadas:", len(corpus))
    if len(corpus):
        print(corpus["fuente"].value_counts().to_string())
    print("\nPáginas de las que no salió ningún par:")
    for item in reporte:
        if item["pares"] == 0:
            print(f"- {item['fuente']}: {item['estado']}")
    return corpus


In [3]:
corpus = main()
corpus.head()


 136 al leer   Hermes


   9 al leer   Ingeniería UECP — FAQ


  31 al leer   SIAPAL


  10 al leer   Ingeniería UECP — Glosario


   0 al leer   Convocatoria abierta 2018


 107 al leer   Bibliotecas — índice


... (23 outputs left out)

   0 al leer   Egresados UNAL   HTTPError: 404 Client Error: Not Found for url: https://egresados.unal.edu.co/preguntas-frecuentes/preguntas-egresados


 105 al leer   Estadística


   8 al leer   Veeduría disciplinaria Medellín
Pares en el corpus, después de separar secciones pegadas: 839
fuente
Ingeniería Bogotá                  161
Hermes                             136
Bibliotecas — índice               105
Estadística                        102
DAMA Manizales                      74
SIAPAL                              31
División salarial                   24
Humanas — Bienestar                 23
Humanas — internacionalización      21
Bibliotecas sede Bogotá             21
Dirección de Personal               19
ORI Medellín                        18
Registro y matrícula Medellín       18
Medicina Bogotá                     14
Gestión vehicular                   12
Ingeniería UECP — Glosario          10
Bibliotecas — CLIC                  10
Ingeniería UECP — FAQ                9
Personal académico                   9
Veeduría disciplinaria Medellín      8
Secretaría General                   7
Explora UN Mundo                     7

Páginas de las que no sal

,pregunta,respuesta,fuente,url,titulo_seccion
0,Recibí un correo para la aceptación del acta d...,Debe realizar la aceptación del acta de inicio...,Hermes,http://www.hermes.unal.edu.co/pages/Consultas/...,
1,¿Cómo puedo conocer la calificación del proyec...,El docente investigador principal debe ingresa...,Hermes,http://www.hermes.unal.edu.co/pages/Consultas/...,
2,¿Cómo puedo saber cuáles fueron los requisitos...,El docente investigador principal debe ingresa...,Hermes,http://www.hermes.unal.edu.co/pages/Consultas/...,
3,¿Cómo se realiza la activación del proyecto?,La activación del proyecto se realiza cuando e...,Hermes,http://www.hermes.unal.edu.co/pages/Consultas/...,
4,"¿Al revisar una solicitud de aval, cómo hago p...","En este caso, es necesario que, a través del r...",Hermes,http://www.hermes.unal.edu.co/pages/Consultas/...,


### **2.3. Revisión de calidad**
---

La revisión no modifica el archivo. Comprueba filas por fuente, vacíos, secciones y la longitud de preguntas y respuestas. Un punto entre iniciales, como en P.A.P.A., no se cuenta como frase sin espacio.


In [4]:
from IPython.display import display

print("Columnas:", list(corpus.columns))
print("Filas:", len(corpus), "| Fuentes con pares:", corpus["fuente"].nunique())
print("Preguntas vacías:", int(corpus["pregunta"].str.strip().eq("").sum()))
print("Respuestas vacías:", int(corpus["respuesta"].str.strip().eq("").sum()))
sin_seccion = corpus["titulo_seccion"].fillna("").str.strip().eq("")
print("Filas sin título de sección:", int(sin_seccion.sum()))
print("\nPares por fuente:")
print(corpus["fuente"].value_counts().to_string())

pegado = corpus["respuesta"].str.contains(r"(?<![A-ZÁÉÍÓÚÑ])[.?!](?=[A-ZÁÉÍÓÚÑ])", regex=True)
url_pegada = corpus["respuesta"].str.contains(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]https?://", regex=True)
print("\nRespuestas con puntuación pegada a la palabra siguiente:", int(pegado.sum()))
print("Respuestas con una URL pegada al texto:", int(url_pegada.sum()))

longitudes = pd.DataFrame({
    "caracteres_pregunta": corpus["pregunta"].str.len(),
    "caracteres_respuesta": corpus["respuesta"].str.len(),
    "palabras_pregunta": corpus["pregunta"].str.split().str.len(),
    "palabras_respuesta": corpus["respuesta"].str.split().str.len(),
})
display(longitudes.describe().round(1))
print("Columnas en disco:", list(pd.read_csv("corpus_faq_unal.csv", nrows=0).columns))


Columnas: ['pregunta', 'respuesta', 'fuente', 'url', 'titulo_seccion']
Filas: 839 | Fuentes con pares: 22
Preguntas vacías: 0
Respuestas vacías: 0
Filas sin título de sección: 587

Pares por fuente:
fuente
Ingeniería Bogotá                  161
Hermes                             136
Bibliotecas — índice               105
Estadística                        102
DAMA Manizales                      74
SIAPAL                              31
División salarial                   24
Humanas — Bienestar                 23
Humanas — internacionalización      21
Bibliotecas sede Bogotá             21
Dirección de Personal               19
ORI Medellín                        18
Registro y matrícula Medellín       18
Medicina Bogotá                     14
Gestión vehicular                   12
Ingeniería UECP — Glosario          10
Bibliotecas — CLIC                  10
Ingeniería UECP — FAQ                9
Personal académico                   9
Veeduría disciplinaria Medellín      8
Secretaría Gen

,caracteres_pregunta,caracteres_respuesta,palabras_pregunta,palabras_respuesta
count,839.0,839.0,839.0,839.0
mean,74.0,477.9,11.7,72.9
std,36.8,451.0,6.0,69.3
min,9.0,22.0,1.0,1.0
25%,50.0,214.0,8.0,32.0
50%,68.0,362.0,11.0,54.0
75%,90.0,598.5,14.0,91.5
max,262.0,5154.0,41.0,783.0


Columnas en disco: ['pregunta', 'respuesta', 'fuente', 'url', 'titulo_seccion']


### **2.4. Lectura del corpus**
---

En esta ejecución el archivo queda con **839 pares** y **22 fuentes**. Las páginas que más aportan son Ingeniería Bogotá (161), Hermes (136), el índice de Bibliotecas (105), Estadística (102) y DAMA Manizales (74). El resto son oficinas y facultades con menos de 35 pares cada una.

Doce páginas de la lista no entran al archivo: nueve responden 404, la Dirección Académica de Bogotá responde 503, la convocatoria de 2018 no trae la FAQ en el HTML y Quejas y reclamos abre una portada sin el listado. Esas direcciones se dejan escritas en la salida de la carga para no presentarlas como si tuvieran texto.

Las preguntas rondan doce palabras (en la revisión, media 11.7 y máximo 41). Las respuestas son mucho más largas y con mucha dispersión: conviene, en la fase 4, comparar la pregunta del usuario con la pregunta del corpus y no solo con el párrafo de la respuesta. Hay filas sin título de sección porque varias páginas no agrupan. Donde sí hay sección —Estadística, Bibliotecas, Ingeniería y el glosario— se conserva.

La revisión marca 17 respuestas con un signo pegado a una mayúscula. Casi todas son listas numeradas (`2.Clave`), siglas (`T.C.`) o comillas, no dos frases sin espacio. No se modifica el archivo por eso.

La normalización y el análisis exploratorio de este desbalance quedan para la fase 2.


## **Créditos**

* **Profesor:** [Felipe Restrepo Calle](https://dis.unal.edu.co/~ferestrepoca/)
* **Asistentes docentes:**
    - [Juan Sebastián Lara Ramírez](https://www.linkedin.com/in/juan-sebastian-lara-ramirez-43570a214/).
* **Diseño de imágenes:**
    - [Rosa Alejandra Superlano Esquibel](mailto:rsuperlano@unal.edu.co).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*
